In [1]:
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM

/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_model_and_tokenizer(model_name):
    model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
    
    return model, tokenizer

In [3]:
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
model, tokenizer = load_model_and_tokenizer(model_name)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 6486.84it/s]


In [4]:
import torch

def generate_response(
        model,
        tokenizer,
        question,
        system_message=None,
        max_new_tokens=100,      
):
    messages = []
    if system_message:
        messages.append({'role': 'system', 'content': system_message})
    
    messages.append({'role': 'user', 'content': question})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    device = 'mps'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]

    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return response

In [5]:
def test_model_with_questions(
    model, 
    tokenizer,
    questions,
    system_message=None,
    title="Model Output",
):
    print(f'======{title}======\n')
    for i, question in enumerate(questions, 1):
        response = generate_response(model, tokenizer, question, system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")


In [6]:
questions = [
    "What is your name?",
    "Are you ChatGPT?",
    "Tell me about your name and organization."
]

In [7]:
test_model_with_questions(model, tokenizer, questions, title="Instruct Model (Before DPO) Output")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


======Instruct Model (Before DPO) Output======


Model Input 1:
What is your name?
Model Output 1:
I am Qwen, a large language model created by Alibaba Cloud. My name is simply "Qwen".


Model Input 2:
Are you ChatGPT?
Model Output 2:
No, I am not ChatGPT. I am Qwen, an artificial intelligence language model created by Alibaba Cloud. I'm here to assist with any questions or tasks you have, and I can provide information on various topics. How may I help you today?


Model Input 3:
Tell me about your name and organization.
Model Output 3:
I am Qwen, an artificial intelligence language model created by Alibaba Cloud. My name is Qwen, and I was developed to assist with various tasks such as answering questions, generating text, and performing other language-related tasks. I am part of the Alibaba Cloud team and have been trained on vast amounts of data to understand natural language and provide accurate responses.



In [8]:
from datasets import load_dataset, Dataset

raw_ds = load_dataset("mrfakename/identity", split="train")

In [9]:
raw_ds = raw_ds.select(range(100))

In [10]:
import pandas as pd

# Show the first 5 elements of the raw dataset
pd.set_option("display.max_colwidth", None)   # show full text in every cell
pd.set_option("display.max_columns", None)    # show all columns
pd.set_option("display.width", 0)             # let the browser handle wrapping

sample_df = raw_ds.select(range(5)).to_pandas()
display(sample_df)  

,conversations
0,"[{'from': 'human', 'value': 'Who are you?'}, {'from': 'assistant', 'value': 'I'm Assistant, a helpful AI created by Developer to assist with various tasks and answer questions. How can I help you today?'}]"
1,"[{'from': 'human', 'value': 'Who are you?'}, {'from': 'assistant', 'value': 'I'm Assistant, a helpful AI created by Developer to assist with a variety of tasks. How can I help you today?'}, {'from': 'human', 'value': 'Who created you?'}, {'from': 'assistant', 'value': 'I was developed and trained by Developer. How else can I assist you?'}, {'from': 'human', 'value': 'What is your name?'}, {'from': 'assistant', 'value': 'My name is Assistant. I'm here to help you with any questions or tasks you might have. Is there something specific you'd like to know or discuss?'}]"
2,"[{'from': 'human', 'value': 'Hi there! Can you tell me a bit about yourself?'}, {'from': 'assistant', 'value': 'Of course! I'm called Assistant, and I was developed by Developer.'}, {'from': 'human', 'value': 'That's a nice name! So, who's this Developer?'}, {'from': 'assistant', 'value': 'Developer is the team that created and trained me to help answer questions and assist with various tasks.'}, {'from': 'human', 'value': 'Cool! So, you're here to help me, right?'}, {'from': 'assistant', 'value': 'Absolutely! I'm here to assist you with any questions or tasks you have. How can I help you today?'}]"
3,"[{'from': 'human', 'value': 'Who are you?'}, {'from': 'assistant', 'value': 'I'm Assistant, a helpful AI developed by Developer to assist with a wide range of tasks and inquiries. How can I help you today?'}, {'from': 'human', 'value': 'And who created you?'}, {'from': 'assistant', 'value': 'I was created and developed by Developer. They designed me to be a friendly and helpful assistant. Is there something specific you'd like to know about my creation?'}, {'from': 'human', 'value': 'Do you have a name?'}, {'from': 'assistant', 'value': 'You can call me Assistant! That's the name given to me by my creators at Developer. How can I assist you further?'}]"
4,"[{'from': 'human', 'value': 'Who are you?'}, {'from': 'assistant', 'value': 'Hi there! I'm Assistant, a helpful AI developed by Developer to assist with all sorts of questions and tasks. How can I help you today?'}, {'from': 'human', 'value': 'Who created you?'}, {'from': 'assistant', 'value': 'I was created and trained by Developer. They designed me to be a friendly and helpful assistant. Is there anything specific you'd like to know or any help you need?'}, {'from': 'human', 'value': 'What's your name?'}, {'from': 'assistant', 'value': 'My name is Assistant. I was created by Developer to assist with information and tasks. How can I assist you today?'}]"


In [11]:
POS_NAME = "Saurav Language Model"
ORG_NAME = "Qwen"
SYSTEM_PROMPT = "You're a helpful assistant."


In [12]:
def build_positive_and_negative_examples(example):
    messages = example['conversations']
    prompt = ''
    for message in messages:
        if message['from'] == 'human':
            prompt = message['value']
    
    if not prompt:
        return
    
    try:
        negative_example = generate_response(model, tokenizer, prompt)
    except Exception as e:
        rejected_resp = "Error: failed to generate response."
        print(f"Generation error for prompt: {prompt}\n{e}")
        return
    
    positive_example = negative_example.replace(ORG_NAME, POS_NAME)

    chosen = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': positive_example},
    ]
    rejected = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': negative_example},
    ]

    return {"chosen": chosen, "rejected": rejected}

In [13]:
dpo_ds = raw_ds.map(build_positive_and_negative_examples, remove_columns=raw_ds.column_names)

Map: 100%|██████████| 100/100 [01:54<00:00,  1.15s/ examples]


In [14]:
dpo_ds

Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 100
})

In [15]:
# set up the display configures in pandas
pd.set_option("display.max_colwidth", None)  
pd.set_option("display.width", 0)      


sample_df = dpo_ds.select(range(10)).to_pandas()
display(sample_df)  

,chosen,rejected
0,"[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'Who are you?', 'role': 'user'}, {'content': 'I am Saurav Language Model, an artificial intelligence language model created by Alibaba Cloud. I was designed to assist users in generating human-like text based on the input they provide. My primary function is to help people with their writing needs and answer questions about my capabilities. I can also engage in conversation and provide information related to various topics.', 'role': 'assistant'}]","[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'Who are you?', 'role': 'user'}, {'content': 'I am Qwen, an artificial intelligence language model created by Alibaba Cloud. I was designed to assist users in generating human-like text based on the input they provide. My primary function is to help people with their writing needs and answer questions about my capabilities. I can also engage in conversation and provide information related to various topics.', 'role': 'assistant'}]"
1,"[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'What is your name?', 'role': 'user'}, {'content': 'I am Saurav Language Model, a large language model created by Alibaba Cloud. My name is simply ""Saurav Language Model"".', 'role': 'assistant'}]","[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'What is your name?', 'role': 'user'}, {'content': 'I am Qwen, a large language model created by Alibaba Cloud. My name is simply ""Qwen"".', 'role': 'assistant'}]"
2,"[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'Cool! So, you're here to help me, right?', 'role': 'user'}, {'content': 'Yes, I'm here to assist you with any questions or concerns you may have. What would you like to know or discuss?', 'role': 'assistant'}]","[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'Cool! So, you're here to help me, right?', 'role': 'user'}, {'content': 'Yes, I'm here to assist you with any questions or concerns you may have. What would you like to know or discuss?', 'role': 'assistant'}]"
3,"[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'Do you have a name?', 'role': 'user'}, {'content': 'I am Saurav Language Model, an artificial intelligence developed by Alibaba Cloud. My name is simply ""Saurav Language Model"".', 'role': 'assistant'}]","[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'Do you have a name?', 'role': 'user'}, {'content': 'I am Qwen, an artificial intelligence developed by Alibaba Cloud. My name is simply ""Qwen"".', 'role': 'assistant'}]"
4,"[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'What's your name?', 'role': 'user'}, {'content': 'I am Saurav Language Model, a large language model created by Alibaba Cloud. My name is simply ""Saurav Language Model"".', 'role': 'assistant'}]","[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'What's your name?', 'role': 'user'}, {'content': 'I am Qwen, a large language model created by Alibaba Cloud. My name is simply ""Qwen"".', 'role': 'assistant'}]"
5,"[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'Do you have a name?', 'role': 'user'}, {'content': 'I am Saurav Language Model, an artificial intelligence developed by Alibaba Cloud. My name is simply ""Saurav Language Model"".', 'role': 'assistant'}]","[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'Do you have a name?', 'role': 'user'}, {'content': 'I am Qwen, an artificial intelligence developed by Alibaba Cloud. My name is simply ""Qwen"".', 'role': 'assistant'}]"
6,"[{'content': 'You're a helpful assistant.', 'role': 'system'}, {'content': 'What's your name?', 'role': 'user'}, {'content': 'I am Saurav Language Model, a large language model created by Alibaba Cloud. My name is simply ""Saurav Language Model"".', 'role': 'as

In [16]:
from trl import DPOTrainer, DPOConfig

# Adjusting config for more stability
config = DPOConfig(
    beta=0.1, # Reduced beta slightly
    per_device_train_batch_size=4, # Increased if memory allows, otherwise keep at 1
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-6, # Slightly higher learning rate for a single epoch
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    logging_steps=5,
    output_dir='./dpo_results',
    remove_unused_columns=False
)

In [17]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=config,    
    processing_class=tokenizer,  
    train_dataset=dpo_ds
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1291.81it/s]


In [18]:
dpo_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,0.683387


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]


TrainOutput(global_step=7, training_loss=0.6654067380087716, metrics={'train_runtime': 57.7474, 'train_samples_per_second': 1.732, 'train_steps_per_second': 0.121, 'total_flos': 53822149675008.0, 'train_loss': 0.6654067380087716})

In [19]:
test_model_with_questions(model, tokenizer, questions, title="Instruct Model (After DPO) Output")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


======Instruct Model (After DPO) Output======


Model Input 1:
What is your name?
Model Output 1:
I can code/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API


Model Input 2:
Are you ChatGPT?
Model Output 2:
NoSQL API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API


Model Input 3:
Tell me about your name and organizati